# Experiment 4 - Stage 1: ECG Expert Pre-training (Anti-Collapse Edition)

**Objective**: Train a 1D-ResNet ECG encoder that produces **diagnostically useful** features (`ecg_encoder.pth`).

**Problem Solved**: 
- Data is **10:1 imbalanced** (negative >> positive). Naive training leads to "All-Negative" collapse (Recall=0).
- Pure SimCLR learns **sample identity** (noise patterns), not **pathological features**.

**Strategy**:
1. **Weighted BCE Loss**: Counter class imbalance by upweighting minority (positive) class.
2. **Focal Loss**: Further focus on hard-to-classify positive samples.  
3. **Supervised-First, Contrastive-Second**: Classification is the **primary** objective (weight=2.0), contrastive is **auxiliary** (weight=0.5).
4. **Validation with F1/Recall Monitoring**: Every epoch checks if the model is stuck in "all-negative" collapse.
5. **Early Warning System**: If Recall=0 for 3+ epochs, automatically increase positive class weight.

**Key Metric**: Recall > 0 (model must detect SOME positive cases).

In [ ]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import pandas as pd
import numpy as np
import wfdb
from tqdm import tqdm
from sklearn.metrics import f1_score, recall_score, precision_score, roc_auc_score, classification_report

# Configuration
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64
EPOCHS = 15
LR = 1e-3
DATA_PATH = r"e:\平时作业\人工智能\data\master_index.csv"
TEMPERATURE = 0.5  # SimCLR Temperature (raised: less aggressive contrastive)

# ==========================================
# Focal Loss: Solves class imbalance better than weighted BCE
# ==========================================
class FocalLoss(nn.Module):
    """
    Focal Loss for imbalanced binary classification.
    Focuses training on hard-to-classify examples (i.e., positive minority).
    gamma=2: standard; alpha=pos_weight for imbalance correction.
    """
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        probs = torch.sigmoid(logits)
        # p_t: probability of correct class
        p_t = probs * targets + (1 - probs) * (1 - targets)
        # alpha_t: weight for each sample
        alpha_t = self.alpha * targets + (1 - targets)
        # Focal modulation
        focal_weight = alpha_t * (1 - p_t) ** self.gamma
        loss = (focal_weight * bce).mean()
        return loss

# ==========================================
# 1D-ResNet Blocks & Model
# ==========================================
class ResBlock1D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=7, stride=stride, padding=3, bias=False)
        self.bn1 = nn.BatchNorm1d(out_ch)
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=7, stride=1, padding=3, bias=False)
        self.bn2 = nn.BatchNorm1d(out_ch)
        self.relu = nn.ReLU(inplace=True)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv1d(in_ch, out_ch, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm1d(out_ch)
            )
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return self.relu(out)

class ECGResNet1D(nn.Module):
    def __init__(self):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=15, stride=2, padding=7, bias=False),
            nn.BatchNorm1d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool1d(kernel_size=3, stride=2, padding=1)
        )
        self.layer1 = ResBlock1D(32, 64, stride=2)
        self.layer2 = ResBlock1D(64, 128, stride=2)
        self.layer3 = ResBlock1D(128, 256, stride=2)
        self.layer4 = ResBlock1D(256, 512, stride=2)
        self.global_pool = nn.AdaptiveAvgPool1d(1)
        
        # Head 1: Classification (PRIMARY task)
        self.classifier = nn.Linear(512, 1)
        
        # Head 2: Contrastive Projection (AUXILIARY task)
        self.projector = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128)
        )
        
        # Head 3: Decoder for Stage2 compatibility
        self.decoder = nn.Sequential(nn.Linear(512, 1024), nn.ReLU(), nn.Linear(1024, 2000))

    def forward(self, x):
        features = self.stem(x)
        features = self.layer1(features)
        features = self.layer2(features)
        features = self.layer3(features)
        features = self.layer4(features)
        latent = self.global_pool(features).squeeze(-1)  # [B, 512]
        
        logits = self.classifier(latent)
        projection = self.projector(latent)
        
        return logits, projection, latent

print(f"Device: {DEVICE}")
print("Model: ECGResNet1D with FocalLoss + Weighted Sampling + Contrastive Auxiliary")

In [ ]:
# ==========================================
# 1. Data Augmentation
# ==========================================
def augment_ecg_signal(raw_full, target_len=2000):
    """ECG augmentation: Random Crop + Scale + Noise"""
    total_len = len(raw_full)
    if total_len > target_len:
        start = np.random.randint(0, total_len - target_len)
        crop = raw_full[start : start + target_len].copy()
    else:
        crop = np.zeros(target_len, dtype=np.float32)
        crop[:min(total_len, target_len)] = raw_full[:min(total_len, target_len)]
    
    scale = np.random.uniform(0.8, 1.2)
    crop = crop * scale
    noise = np.random.normal(0, 0.05, size=crop.shape)
    crop = crop + noise
    return crop.astype(np.float32)

class ContrastiveECGDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop=True)
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        ecg_path = row['ecg_path']
        label = float(row['label'])
        
        raw_sig = np.zeros(2000, dtype=np.float32)
        if pd.notna(ecg_path):
            try:
                sig, _ = wfdb.rdsamp(ecg_path, channels=[0])
                if sig is not None:
                    raw_sig = sig.flatten()
                    raw_sig = (raw_sig - np.mean(raw_sig)) / (np.std(raw_sig) + 1e-6)
            except:
                pass
        
        view1 = augment_ecg_signal(raw_sig)
        view2 = augment_ecg_signal(raw_sig)
        
        return (
            torch.tensor(view1, dtype=torch.float32).unsqueeze(0),
            torch.tensor(view2, dtype=torch.float32).unsqueeze(0)
        ), torch.tensor(label, dtype=torch.float32)

# ==========================================
# 2. NT-Xent Loss (SimCLR)
# ==========================================
class NTXentLoss(nn.Module):
    def __init__(self, temperature=0.5):
        super().__init__()
        self.temperature = temperature
        self.criterion = nn.CrossEntropyLoss()
        
    def forward(self, z_i, z_j):
        batch_size = z_i.size(0)
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        z_all = torch.cat([z_i, z_j], dim=0)
        sim_matrix = torch.matmul(z_all, z_all.T) / self.temperature
        mask = torch.eye(2 * batch_size, device=z_i.device).bool()
        sim_matrix.masked_fill_(mask, -9e15)
        labels = torch.cat([
            torch.arange(batch_size, device=z_i.device) + batch_size,
            torch.arange(batch_size, device=z_i.device)
        ], dim=0)
        return self.criterion(sim_matrix, labels)

# ==========================================
# 3. Data Preparation with Class Balance
# ==========================================
df_master = pd.read_csv(DATA_PATH)
df_valid_ecg = df_master[df_master['has_ecg'] == True].sample(frac=1.0, random_state=42).reset_index(drop=True)

# Analyze class distribution
n_pos = int(df_valid_ecg['label'].sum())
n_neg = len(df_valid_ecg) - n_pos
pos_ratio = n_pos / len(df_valid_ecg)
pos_weight = n_neg / max(n_pos, 1)  # Inverse frequency weight

print(f"Total ECG samples: {len(df_valid_ecg)}")
print(f"  Positive (HF risk): {n_pos} ({pos_ratio*100:.1f}%)")
print(f"  Negative (Normal):  {n_neg} ({(1-pos_ratio)*100:.1f}%)")
print(f"  Imbalance Ratio: 1:{pos_weight:.1f}")
print(f"  Computed pos_weight for loss: {pos_weight:.2f}")

# Train/Val Split (80/20)
split_idx = int(0.8 * len(df_valid_ecg))
train_df = df_valid_ecg.iloc[:split_idx].reset_index(drop=True)
val_df = df_valid_ecg.iloc[split_idx:].reset_index(drop=True)

# ==========================================
# CRITICAL: Weighted Random Sampler
# Forces each batch to have ~50% positive samples
# ==========================================
train_labels = train_df['label'].values
sample_weights = np.where(train_labels == 1, pos_weight, 1.0)
sampler = WeightedRandomSampler(
    weights=sample_weights, 
    num_samples=len(train_df), 
    replacement=True
)

train_dataset = ContrastiveECGDataset(train_df)
val_dataset = ContrastiveECGDataset(val_df)

# Use sampler instead of shuffle!
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

print(f"\nTrain: {len(train_df)} | Val: {len(val_df)}")
print(f"Using WeightedRandomSampler to balance batches.")

In [ ]:
# ==========================================
# 4. Training Loop with Anti-Collapse Monitoring
# ==========================================

model = ECGResNet1D().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)

# Loss Functions
# PRIMARY: Focal Loss with alpha = pos_weight (counter imbalance)
criterion_cls = FocalLoss(alpha=pos_weight, gamma=2.0)
# AUXILIARY: Contrastive (lower temperature = less aggressive)
criterion_con = NTXentLoss(temperature=TEMPERATURE)

# Loss Weights: Classification is DOMINANT
WEIGHT_CLS = 2.0   # Classification is the primary objective
WEIGHT_CON = 0.3    # Contrastive is auxiliary

# Anti-Collapse Tracking
collapse_counter = 0
best_val_f1 = 0.0

print("=" * 60)
print("🚀 ECG Pre-training: Supervised-First + Contrastive-Auxiliary")
print(f"   Loss = {WEIGHT_CLS}×Focal + {WEIGHT_CON}×NT-Xent")
print(f"   Focal Loss alpha (pos_weight): {pos_weight:.2f}, gamma: 2.0")
print(f"   Anti-Collapse: Monitor Recall every epoch")
print("=" * 60)

for epoch in range(EPOCHS):
    # ---- TRAIN ----
    model.train()
    total_loss = 0
    total_cls = 0
    total_con = 0
    train_preds = []
    train_targets = []
    
    for (view1, view2), labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
        labels = labels.to(DEVICE).unsqueeze(1)
        
        optimizer.zero_grad()
        
        logits1, proj1, _ = model(view1)
        logits2, proj2, _ = model(view2)
        
        # Classification Loss (Focal)
        loss_cls = 0.5 * (criterion_cls(logits1, labels) + criterion_cls(logits2, labels))
        
        # Contrastive Loss
        loss_con = criterion_con(proj1, proj2)
        
        # Weighted Combination: Classification DOMINATES
        loss = WEIGHT_CLS * loss_cls + WEIGHT_CON * loss_con
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        total_cls += loss_cls.item()
        total_con += loss_con.item()
        
        # Track predictions for train metrics
        with torch.no_grad():
            avg_logits = 0.5 * (logits1 + logits2)
            preds = (torch.sigmoid(avg_logits) > 0.5).float()
            train_preds.extend(preds.cpu().numpy().flatten())
            train_targets.extend(labels.cpu().numpy().flatten())
    
    # ---- VALIDATE ----
    model.eval()
    val_preds = []
    val_targets = []
    val_probs = []
    
    with torch.no_grad():
        for (view1, view2), labels in val_loader:
            view1 = view1.to(DEVICE)
            labels_np = labels.numpy()
            
            logits, _, _ = model(view1)
            probs = torch.sigmoid(logits).cpu().numpy().flatten()
            preds = (probs > 0.5).astype(int)
            
            val_probs.extend(probs)
            val_preds.extend(preds)
            val_targets.extend(labels_np)
    
    val_preds = np.array(val_preds)
    val_targets = np.array(val_targets)
    val_probs = np.array(val_probs)
    train_preds = np.array(train_preds)
    train_targets = np.array(train_targets)
    
    # ---- METRICS ----
    try:
        val_f1 = f1_score(val_targets, val_preds, zero_division=0)
        val_recall = recall_score(val_targets, val_preds, zero_division=0)
        val_prec = precision_score(val_targets, val_preds, zero_division=0)
        val_auc = roc_auc_score(val_targets, val_probs)
    except:
        val_f1 = 0; val_recall = 0; val_prec = 0; val_auc = 0.5
    
    train_recall = recall_score(train_targets, train_preds, zero_division=0)
    
    # Positive prediction stats
    n_val_pos_pred = int(val_preds.sum())
    n_val_total = len(val_preds)
    
    # ---- LOGGING ----
    avg_loss = total_loss / len(train_loader)
    avg_cls = total_cls / len(train_loader)
    avg_con = total_con / len(train_loader)
    
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    print(f"  Loss: Total={avg_loss:.4f} | Focal={avg_cls:.4f} | Con={avg_con:.4f}")
    print(f"  Val:  AUC={val_auc:.4f} | F1={val_f1:.4f} | Prec={val_prec:.4f} | Recall={val_recall:.4f}")
    print(f"  Val Predictions: {n_val_pos_pred}/{n_val_total} predicted positive ({n_val_pos_pred/max(n_val_total,1)*100:.1f}%)")
    
    # ---- ANTI-COLLAPSE CHECK ----
    if val_recall == 0:
        collapse_counter += 1
        print(f"  ⚠️  COLLAPSE WARNING ({collapse_counter}/3): Recall=0! Model predicts ALL NEGATIVE.")
        
        if collapse_counter >= 3:
            # Emergency: dramatically increase classification weight
            WEIGHT_CLS *= 1.5
            # Also adjust Focal alpha
            criterion_cls = FocalLoss(alpha=pos_weight * 2.0, gamma=2.0)
            print(f"  🔧 AUTO-FIX: Increased WEIGHT_CLS to {WEIGHT_CLS:.1f}, doubled Focal alpha")
            collapse_counter = 0  # Reset counter
    else:
        collapse_counter = 0  # Reset if model is producing positive predictions
        print(f"  ✅ Model is detecting positives (no collapse)")
    
    # ---- SAVE BEST ----
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "ecg_encoder.pth")
        print(f"  💾 New best model saved! F1={val_f1:.4f}")

# ---- FINAL REPORT ----
print("\n" + "=" * 60)
print("📊 Final ECG Pre-training Report")
print("=" * 60)

# Run final evaluation
model.eval()
final_preds = []
final_targets = []
final_probs = []

with torch.no_grad():
    for (view1, _), labels in val_loader:
        view1 = view1.to(DEVICE)
        logits, _, _ = model(view1)
        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        final_probs.extend(probs)
        final_preds.extend((probs > 0.5).astype(int))
        final_targets.extend(labels.numpy())

final_preds = np.array(final_preds)
final_targets = np.array(final_targets)
final_probs = np.array(final_probs)

print(classification_report(final_targets, final_preds, 
                           target_names=['Normal (0)', 'HF Risk (1)'], 
                           zero_division=0))

try:
    final_auc = roc_auc_score(final_targets, final_probs)
    print(f"AUC: {final_auc:.4f}")
except:
    print("AUC: N/A")

final_recall = recall_score(final_targets, final_preds, zero_division=0)
if final_recall == 0:
    print("\n❌ CRITICAL: Final Recall = 0. The ECG encoder is USELESS for fusion.")
    print("   Recommendations:")
    print("   1. Increase EPOCHS to 20+")
    print("   2. Set WEIGHT_CLS = 5.0 (more classification pressure)")
    print("   3. Reduce WEIGHT_CON = 0.1 (less contrastive distraction)")
    print("   4. Check if ECG data actually contains discriminative signal")
elif final_recall < 0.3:
    print(f"\n⚠️  WARNING: Final Recall = {final_recall:.3f} (low).")
    print("   The encoder works but weakly. Consider more epochs.")
else:
    print(f"\n✅ ECG Encoder is HEALTHY. Recall={final_recall:.3f}, will contribute to fusion.")

print(f"\nBest model saved: ecg_encoder.pth (Best Val F1: {best_val_f1:.4f})")